# P22 — paired RNA+ATAC workflow and evidence

**Research outcome: inconclusive. This notebook does not claim a new biological result.**

This is one entry point for the current workflow: software tests, source preflight, saved RNA findings, paired-input requirements, offline HEAD examples, reviewed code, and next steps. It uses the repository's existing implementations instead of maintaining a second copy.

## Run safely

Use this notebook inside the accepted P22 checkout with its existing Python 3.11 environment. Select that kernel, then **Restart Kernel and Run All**. No installation cell is included. A downloaded notebook alone needs the accompanying checkout to run.

The first working cell runs the offline parser/capture/preflight tests. Later cells stop if those tests fail. Temporary test fixtures are allowed. No cell downloads datasets, invokes a live transport, starts Docker, performs real model training, changes approvals, or overwrites the old RNA notebook. Existing real-analysis environment flags do not enable those actions here.

Saved RNA findings are displayed, **not recomputed**. Synthetic header examples are **not observations from GEO**. Local Jupyter kernel traffic is not a dataset request.

The complete historical RNA workflow remains in [the original notebook](../P22_down_syndrome_all_in_one.ipynb). Its live-data modes are separate from this offline notebook.

In [1]:
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path
from uuid import uuid4

from IPython.display import Code, Markdown, display

assert sys.version_info[:2] == (3, 11), "Select the existing P22 Python 3.11 kernel."
sys.dont_write_bytecode = True
cwd = Path.cwd().resolve()
REPO = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "scripts/launcher_head_capture.py").is_file()
     and (path / "pyproject.toml").is_file()),
    None,
)
assert REPO is not None, "Run this notebook from the accepted P22 checkout."
OBSERVED_AT = datetime.now(UTC).isoformat()
print(json.dumps({
    "python": platform.python_version(), "executable": sys.executable,
    "repository": str(REPO), "observed_at": OBSERVED_AT,
    "mode": "OFFLINE_EVIDENCE_AND_TESTS", "live_enabled": False,
}, indent=2))

{
  "python": "3.11.16",
  "executable": "/Users/anuragdani/Github/niw-eb1a/P22/.venv-p22/bin/python",
  "repository": "/private/tmp/p22-input-feasibility.WhUFn3/workspace",
  "observed_at": "2026-09-15T04:23:15.127126+00:00",
  "mode": "OFFLINE_EVIDENCE_AND_TESTS",
  "live_enabled": false
}


In [2]:
# Test first. This list excludes this notebook's own kernel test to avoid recursion.
test_command = [
    sys.executable, "-m", "pytest", "-q",
    "tests/test_development_head.py",
    "tests/test_development_head_capture.py",
    "tests/test_launcher_head_capture.py",
]
test_environment = dict(os.environ, PYTHONPATH=str(REPO / "src"),
                        PYTHONDONTWRITEBYTECODE="1", PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
test_run = subprocess.run(
    test_command, cwd=REPO, env=test_environment,
    capture_output=True, text=True, timeout=120, check=False,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
OFFLINE_TESTS_PASSED = test_run.returncode == 0
assert OFFLINE_TESTS_PASSED, "Offline tests failed. Stop; do not proceed to later cells."

........................................................................ [ 39%]
........................................................................ [ 79%]
.....................................                                    [100%]
181 passed in 0.41s



## 1. Observe the current local inputs

The preflight examines four pinned files, file/path safety, the output destination, and disk headroom. It does not reserve that destination or prove runtime controls. Insufficient disk or changed evidence is a refusal, not a reason to change the limits.

In [3]:
assert OFFLINE_TESTS_PASSED
sys.path.insert(0, str(REPO / "scripts"))
import launcher_head_capture

assert Path(launcher_head_capture.__file__).resolve() == REPO / "scripts/launcher_head_capture.py"
# Do not create this path. Its absence is part of the preflight contract.
output_candidate = REPO / "reports/generated" / ("notebook-observation-" + uuid4().hex)
preflight_result = launcher_head_capture.preflight(
    repo=REPO, output_dir=output_candidate, observed_at=OBSERVED_AT,
)
print(json.dumps(preflight_result, indent=2))
assert preflight_result["status"] == "OFFLINE_PREFLIGHT_OK", "Preflight refused. Stop here."
assert preflight_result["live_allowed"] is False
assert preflight_result["runtime_controls"] == "UNVERIFIED"
assert preflight_result["output_reserved"] is False
assert not output_candidate.exists()

{
  "status": "OFFLINE_PREFLIGHT_OK",
  "reason": null,
  "observed_at": "2026-09-15T04:23:15.127126+00:00",
  "source_hashes": {
    "configs/development_object_source_contract.json": "9a13d8beafa5e00d81f8f985649a6a466c6b4fe3ebf008206f840c32316113f2",
    "docs/PAIRED_MULTIOME_REMAINING_EVIDENCE_2026-09-08.md": "66b1d8158f5b6c4260085cddb736db2d59de9df610e5dcac5ccfc2c4f63b18b9",
    "scripts/capture_development_head.py": "335d35c51b27d48704eaad62a695d74e6e91f456e3148b15abe8d3aff846029c",
    "scripts/validate_development_head.py": "c761489202fdbfaba5153f7a508aefa19028cf87f681bd473382c6c4c713ac9d"
  },
  "disk_free_bytes": 22397607936,
  "requests_attempted": 0,
  "body_bytes": 0,
  "live_allowed": false,
  "runtime_controls": "UNVERIFIED",
  "output_reserved": false,
  "scientific_gate_effect": "NONE"
}


## 2. The selected source and unchanged resource limits

The candidate URL is a stored proposal, not a verified download link from this run. The object contents, decoded size, full-load memory, and applicable assay classes remain unknown. The limits for a proposed future HEAD do not authorize an object download.

In [4]:
contract = json.loads((REPO / "configs/development_object_source_contract.json").read_text())
assert contract["status"] == "SOURCE_UNRESOLVED"
assert contract["object_inspection_authorized"] is False
assert contract["current_object_permissions"] == {
    "network_bytes": 0, "decoded_bytes": 0, "output_bytes": 0,
    "execution_seconds": 0, "training_allowed": False,
}
print(json.dumps({
    "status": contract["status"],
    "candidate_payload_url": contract["candidate_payload_url"],
    "candidate_url_basis": contract["candidate_url_basis"],
    "exact_payload_bytes": contract["exact_payload_bytes"],
    "expected_decoded_bytes": contract["expected_decoded_bytes"],
    "expected_peak_memory_bytes": contract["expected_peak_memory_bytes"],
    "current_object_permissions": contract["current_object_permissions"],
    "reader_proof": contract["reader_proof"],
    "future_HEAD_proposal_only": contract["single_next_action_proposal"],
}, indent=2))

{
  "status": "SOURCE_UNRESOLVED",
  "candidate_payload_url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305146/suppl/GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz",
  "candidate_url_basis": "Derived from the pinned source note's directory and exact filename; not a new HTTP observation",
  "exact_payload_bytes": null,
  "expected_decoded_bytes": null,
  "expected_peak_memory_bytes": null,
  "current_object_permissions": {
    "network_bytes": 0,
    "decoded_bytes": 0,
    "output_bytes": 0,
    "execution_seconds": 0,
    "training_allowed": false
  },
  "reader_proof": {
    "image_id": "sha256:264df259ebb2fa4ff2a3db6d346ec273e18d4c84af727b1cc4b39065a9e8edbb",
    "r_version": "4.6.1",
    "matrix_version": "1.7.6",
    "tested_serialization": "saveRDS/readRDS of a 392-byte .rds fixture; not save/load of a .rda workspace",
    "tested_classes": [
      "list",
      "dgCMatrix",
      "data.frame"
    ],
    "fixture_report_sha256": "1cf56325eb4f08c86d11661b59ec99

## 3. Completed RNA evidence — saved results, not a new run

These two sections reproduce the outcome sections from the unchanged, hash-checked research reports. Software test counts are not biological sample sizes. Reproducibility does not turn an inconclusive result into a positive result.

In [5]:
record_pins = {
    "docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md":
        "81bb3c313749f4b2751e6259a0c52544d7b10762baddc80597abd36521b96354",
    "docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md":
        "a8892ffd454898bcdc5594c79e09d55d07d70398be5ce3f8f9b58cde563edd36",
}
saved_records = {}
for relative, expected in record_pins.items():
    with (REPO / relative).open("rb") as handle:
        raw = handle.read(65537)
    assert len(raw) <= 65536, "Saved record exceeds the notebook read limit."
    assert hashlib.sha256(raw).hexdigest() == expected, "Saved RNA evidence changed."
    saved_records[relative] = raw.decode("utf-8")
    print(relative, expected)

rna_report = saved_records["docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md"]
influence_report = saved_records["docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md"]
assert "## Final evidence package" in rna_report
assert "## Execution and verification" in influence_report
display(Markdown(rna_report.split("## Final evidence package")[0]))
display(Markdown(influence_report.split("## Execution and verification")[0]))

docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md 81bb3c313749f4b2751e6259a0c52544d7b10762baddc80597abd36521b96354
docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md a8892ffd454898bcdc5594c79e09d55d07d70398be5ce3f8f9b58cde563edd36


# External RNA replication: real-data results

Completed locally on 2026-09-08 (run timestamps are 2026-09-09 UTC).
Fresh Colab verification and final archive comparison completed 2026-09-09.

## Outcome

All four frozen comparisons and the headline are **inconclusive**. The code and
real experiment and all five original RNA plan tasks are complete. Fresh Colab
executions, archived hashes and local result comparison passed. This is an
actual summary-effect comparison, not a synthetic demonstration. It neither
establishes directional replication nor proves that reproducible effects are absent.

Each comparison used nine DS and eight control discovery donors, with exact PCW
and sex adjustment. The external publication has five DS and five control donors;
only its published differential-expression summaries were used.

| Discovery → external | Cells | Shared genes | Spearman rho (95% donor-bootstrap CI) | Top-100 direction agreement | Upper permutation p | Outcome |
|---|---:|---:|---|---:|---:|---|
| RG → oRG | 16,855 | 9,051 | -0.0079 (-0.2516, 0.3187) | 58% | 0.01499 | Inconclusive |
| RG → vRG | 16,855 | 8,572 | -0.0305 (-0.2425, 0.2644) | 53% | 0.18282 | Inconclusive |
| RG_prol ∪ IPC_prol → CP | 8,417 | 9,213 | 0.0967 (-0.1806, 0.3349) | 59% | 0.04096 | Inconclusive |
| IPC → IP | 10,621 | 7,857 | 0.0097 (-0.1920, 0.2642) | 53% | 0.10390 | Inconclusive |

Each row completed 1,000 class-stratified donor bootstraps and 1,000 whole-vector
permutations, with zero failed bootstraps. The oRG agreement p-value alone does
not satisfy the frozen support rule: its rho interval crosses zero. RG cells
appear in two comparisons and must not be counted twice as independent evidence.
All lower-tail p-values, agreement intervals and audit fields are in the CSV.

The [frozen plan](EXTERNAL_RNA_REPLICATION_PLAN.md) defines gene exclusions,
expression floors, top-gene reranking, resampling and descriptive headline rules.
The headline rule carries no guaranteed 2.5% family-wise error calibration.



# RNA donor influence: results, 2026-09-10

POST_HOC_EXPLORATORY. This completes the bounded F4–F5 RNA diagnostic, not the
full P22 plan. Original RNA results remain inconclusive; G8 is unchanged.
No paired predictive evaluation or real neural training ran.

## Result

All four saved baselines reproduced exactly, including donor order and saved
gene/donor/cell counts. Every donor was omitted once per comparison: 68/68
omissions available, each with design rank 4 and 12 residual degrees of freedom.

`PCW18_DS_16545` had the largest absolute same-support influence in each comparison.
The same donors contribute to every comparison; RG comparisons also share cells.
These are not four independent observations about that donor.

| Comparison | Baseline rho | Same-support delta omitting this donor | Omission rho | Share of absolute influence |
|---|---:|---:|---:|---:|
| RG → oRG | -0.007896 | +0.191714 | 0.183818 | 20.0% |
| RG → vRG | -0.030550 | +0.165192 | 0.134642 | 19.5% |
| cycling progenitors → CP | 0.096712 | +0.169408 | 0.266120 | 20.4% |
| IPC → IP | 0.009666 | +0.118950 | 0.128616 | 16.0% |

These shares describe concentration; they are not significance thresholds.
For this donor, the support delta was zero in all four comparisons. Across all
68 omissions, the largest absolute support delta was 0.000756456. At most five
baseline genes were lost and at most one new gene became eligible per omission.
New genes were counted but excluded from the diagnostic endpoint.

Decision: keep all donors and preserve the original conclusion. Do not select
donor exclusions to improve agreement. External-donor uncertainty remains
unresolved. Advance the separately planned paired-input feasibility tasks before
proposing real training; this diagnostic supplies neither common measured ATAC
counts nor accepted external QC.



## 4. Paired-input and training status

The raw-library pilot preserved paired counts for its tested donor/library slice. That is not an accepted development cohort. The two tested library peak lists had zero exact common regions; that result does not prove that every possible processed object is unusable.

External release/QC semantics and compatible measured ATAC counts remain unresolved. Known study provenance is not a complete specimen-independence audit. Synthetic training, final refits, and one-shot scoring already exercised the software; no real external multimodal evaluation followed.

The authoritative criteria below come from the unchanged object contract. Each criterion must apply to the actual selected cohorts. Missing regions cannot become fabricated zero counts.

In [6]:
display(Markdown("\n".join(
    "- " + requirement
    for requirement in contract["scientific_acceptance_requirements_unchanged"]
)))
print("Real paired training allowed:", contract["current_object_permissions"]["training_allowed"])

- paired raw RNA and measured ATAC counts with exact retained barcode/donor join
- release and QC/count-stage semantics
- age/class support and specimen provenance
- genome build, interval coordinates and count units
- exact common measured regions; never zero-fill unmatched peaks
- feature-selection provenance including discovery donors/labels; cohort-wide cluster peaks are not automatically label-independent or train-fold-only, even when disclosed
- post-QC donor/class support for five repeated five-fold internal evaluation and inner validation; two donors per class are insufficient and 13+13 is external margin context, not a development minimum
- separate development, external and cross-cohort decisions

Real paired training allowed: False


## 5. Offline examples of the accepted HEAD code

The following header is an invented fixture from `example.invalid`. Its length of 123 bytes has no connection to the real candidate payload. The parser's internal `SOURCE_METADATA_OBSERVED` label describes that fixture only.

The capture-core call supplies **no transport**. It must refuse before any request. Notebook success cannot enable a live connection.

In [7]:
from capture_development_head import PINNED_URL, capture_head
from validate_development_head import parse_head_evidence

synthetic_header = b"HTTP/1.1 200 OK\r\nContent-Length: 123\r\n\r\n"
demo_metadata = parse_head_evidence(
    synthetic_header, url="https://example.invalid/synthetic.rda.gz",
    observed_at="SYNTHETIC_FIXTURE",
)
assert demo_metadata["declared_payload_bytes"] == 123
assert demo_metadata["publisher_payload_checksum"] is None
print(json.dumps({"evidence_kind": "SYNTHETIC_DEMO_NOT_GEO_OBSERVATION",
                  "parser_result": demo_metadata}, indent=2))

capture_refusal = capture_head(url=PINNED_URL, observed_at=OBSERVED_AT)
assert capture_refusal["capture_status"] == "REFUSED"
assert capture_refusal["reason"] == "LIVE_TRANSPORT_NOT_ENABLED"
assert capture_refusal["requests_attempted"] == 0
assert capture_refusal["body_bytes"] == 0
print(json.dumps(capture_refusal, indent=2))

bad_label = launcher_head_capture.preflight(
    repo=REPO, output_dir=output_candidate, observed_at="bad\nlabel",
)
assert bad_label["reason"] == "BAD_LABEL"
assert bad_label["source_hashes"] == {}
assert bad_label["disk_free_bytes"] is None
print("Invalid-label refusal:", bad_label["reason"])

{
  "evidence_kind": "SYNTHETIC_DEMO_NOT_GEO_OBSERVATION",
  "parser_result": {
    "record_kind": "development_head_metadata",
    "status": "SOURCE_METADATA_OBSERVED",
    "url": "https://example.invalid/synthetic.rda.gz",
    "observed_at": "SYNTHETIC_FIXTURE",
    "http_status": 200,
    "header_record_sha256": "ee08bb20e8232344af9b5e60563240e74e4d9ab6206ba0c157f55cc9fcbed7cd",
    "header_bytes": 40,
    "declared_payload_bytes": 123,
    "etag": null,
    "last_modified": null,
    "accept_ranges": null,
    "publisher_payload_checksum": null,
    "body_bytes": 0,
    "scientific_gate_effect": "NONE"
  }
}
{
  "capture_status": "REFUSED",
  "reason": "LIVE_TRANSPORT_NOT_ENABLED",
  "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305146/suppl/GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz",
  "observed_at": "2026-09-15T04:23:15.127126+00:00",
  "requests_attempted": 0,
  "header_bytes": 0,
  "raw_header_base64": "",
  "header_record_sha256": null,
  "metadata"

## 6. Exact implementation used by this notebook

The three current scripts are displayed below. They remain the source of truth; notebook cells import them. Open the collapsed output to inspect the code. The preflight has no live launcher or CLI.

In [8]:
for relative in (
    "scripts/launcher_head_capture.py",
    "scripts/capture_development_head.py",
    "scripts/validate_development_head.py",
):
    display(Markdown("### " + relative))
    display(Code(filename=str(REPO / relative), language="python"))

### scripts/launcher_head_capture.py

"""P22 launcher slice: read-only offline preflight observation.

Diagnostic-only. No timing, race, or native-control proof. Output path is not
reserved; future launcher must revalidate and own race/resource/authorization
safeguards. Not a live runner.
"""

from __future__ import annotations

import hashlib
import os
import shutil
import stat
from pathlib import Path

PINNED_FILES: tuple[tuple[str, str, int], ...] = (
    (
        "configs/development_object_source_contract.json",
        "9a13d8beafa5e00d81f8f985649a6a466c6b4fe3ebf008206f840c32316113f2",
        6429,
    ),
    (
        "docs/PAIRED_MULTIOME_REMAINING_EVIDENCE_2026-09-08.md",
        "66b1d8158f5b6c4260085cddb736db2d59de9df610e5dcac5ccfc2c4f63b18b9",
        14597,
    ),
    (
        "scripts/capture_development_head.py",
        "335d35c51b27d48704eaad62a695d74e6e91f456e3148b15abe8d3aff846029c",
        7088,
    ),
    (
        "scripts/validate_development_head.py",
        "c761489202fdbfaba5153f7a508aefa19028cf87f681bd473382c6c4c713ac9d",
        4753,
    ),
)
MAX_FILE_SIZE = 65536
MIN_DISK_FREE = 10737418240


def _bad_label(observed_at: object) -> bool:
    if not isinstance(observed_at, str):
        return True
    if not 1 <= len(observed_at) <= 64:
        return True
    return any(ord(ch) < 32 or 127 <= ord(ch) <= 159 for ch in observed_at)


def _safe_path(p: Path) -> bool:
    if not p.is_absolute():
        return False
    return ".." not in p.parts and "\0" not in str(p)


def _nonsymlink_ancestors(p: Path) -> bool:
    cur = p
    while True:
        try:
            if cur.is_symlink():
                return False
        except OSError:
            return False
        parent = cur.parent
        if parent == cur:
            return True
        cur = parent


def _check_pinned_file(repo: Path, rel: str, expected: str, expected_size: int) -> str | None:
    target = repo / rel
    if not _nonsymlink_ancestors(target):
        return "UNSAFE_PATH"
    try:
        flags = os.O_RDONLY | os.O_NOFOLLOW | os.O_NONBLOCK
    except AttributeError:
        return "INPUT_UNAVAILABLE"
    try:
        fd = os.open(target, flags)
        try:
            st = os.fstat(fd)
            if not stat.S_ISREG(st.st_mode):
                return "UNSAFE_PATH"
            if st.st_size > MAX_FILE_SIZE:
                return "FILE_TOO_LARGE"
            if st.st_size != expected_size:
                return "PIN_MISMATCH"
            data = bytearray()
            while len(data) <= MAX_FILE_SIZE:
                chunk = os.read(fd, MAX_FILE_SIZE + 1 - len(data))
                if not chunk:
                    break
                data.extend(chunk)
            if len(data) > MAX_FILE_SIZE:
                return "FILE_TOO_LARGE"
            if len(data) != expected_size or hashlib.sha256(data).hexdigest() != expected:
                return "PIN_MISMATCH"
            return None
        finally:
            os.close(fd)
    except OSError:
        return "INPUT_UNAVAILABLE"


def preflight(*, repo: Path, output_dir: Path, observed_at: str) -> dict:
    result: dict = {
        "status": "REFUSED",
        "reason": None,
        "observed_at": None,
        "source_hashes": {},
        "disk_free_bytes": None,
        "requests_attempted": 0,
        "body_bytes": 0,
        "live_allowed": False,
        "runtime_controls": "UNVERIFIED",
        "output_reserved": False,
        "scientific_gate_effect": "NONE",
    }

    if _bad_label(observed_at):
        result["reason"] = "BAD_LABEL"
        return result
    result["observed_at"] = observed_at

    if not isinstance(repo, Path) or not isinstance(output_dir, Path):
        result["reason"] = "UNSAFE_PATH"
        return result
    if not _safe_path(repo) or not _safe_path(output_dir):
        result["reason"] = "UNSAFE_PATH"
        return result
    try:
        if not _nonsymlink_ancestors(repo) or not repo.is_dir():
            result["reason"] = "UNSAFE_PATH"
            return result



### scripts/capture_development_head.py

"""Capture one pinned HEAD header block through an explicit trusted transport.

No default live transport or CLI. The separately authorized launcher must verify
source hashes, TLS configuration, isolation, memory/swap/disk/output controls
and a hard process watchdog. Python signals can be delayed by C execution.
body_bytes counts application reads only; TLS/OS buffering is not measured here.
"""

import base64
import hashlib
import signal as _signal
import threading
from time import monotonic as _monotonic

from validate_development_head import parse_head_evidence

PINNED_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305146/suppl/"
    "GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz"
)
PINNED_HOST = "ftp.ncbi.nlm.nih.gov"
DEADLINE_SECONDS = 15.0
MAX_HEADER_BYTES = 65536


def _valid_label(value):
    return (
        isinstance(value, str)
        and bool(value)
        and not any(ord(ch) < 32 or 127 <= ord(ch) <= 159 for ch in value)
    )


def _report(reason, url, observed_at, raw=b"", attempts=0, metadata=None):
    return {
        "capture_status": "REFUSED" if reason else "CAPTURED",
        "reason": reason,
        "url": url,
        "observed_at": observed_at,
        "requests_attempted": attempts,
        "header_bytes": len(raw),
        "raw_header_base64": base64.b64encode(raw).decode("ascii"),
        "header_record_sha256": hashlib.sha256(raw).hexdigest() if raw else None,
        "metadata": None if reason else metadata,
        "body_bytes": 0,
        "scientific_gate_effect": "NONE",
    }


def capture_head(*, url: str, observed_at: str, connection_factory=None) -> dict:
    """Return captured metadata or a bounded-category refusal, never authorization.

    The explicit factory constructs an HTTPSConnection-like object without I/O;
    request() performs connection/TLS/send. A factory that raises before returning
    owns cleanup of resources it never handed over. The launcher must supply a
    verified TLS transport without proxy/retry behavior. No arbitrary URL is allowed.
    """
    safe_url = url if isinstance(url, str) and url == PINNED_URL else None
    safe_time = observed_at if _valid_label(observed_at) else None
    if safe_url is None or safe_time is None:
        return _report("BAD_TARGET" if safe_url is None else "BAD_LABEL", safe_url, safe_time)
    if connection_factory is None:
        return _report("LIVE_TRANSPORT_NOT_ENABLED", safe_url, safe_time)

    deadline = _monotonic() + DEADLINE_SECONDS

    def remaining():
        seconds = deadline - _monotonic()
        if seconds <= 0:
            raise TimeoutError("capture deadline")
        return seconds

    # Fail before transport if another timer/handler cannot safely be preserved.
    try:
        if (
            threading.current_thread() is not threading.main_thread()
            or not all(
                callable(getattr(_signal, name, None))
                for name in ("getsignal", "signal", "getitimer", "setitimer")
            )
            or _signal.getitimer(_signal.ITIMER_REAL) != (0, 0)
        ):
            return _report("DEADLINE_UNAVAILABLE", safe_url, safe_time)
        previous_handler = _signal.getsignal(_signal.SIGALRM)
        if not callable(previous_handler) and previous_handler not in (
            _signal.SIG_DFL,
            _signal.SIG_IGN,
        ):
            return _report("DEADLINE_UNAVAILABLE", safe_url, safe_time)
    except Exception:
        return _report("DEADLINE_UNAVAILABLE", safe_url, safe_time)

    def alarm(signum, frame):
        raise TimeoutError("capture deadline")

    conn = None
    attempts = 0
    raw = bytearray()
    reason = None
    metadata = None
    cleanup_reason = None
    try:
        try:
            _signal.signal(_signal.SIGALRM, alarm)
            _signal.setitimer(_signal.ITIMER_REAL, remaining())
        except TimeoutError:
            reason = "DEADLINE_EXPIRED"
        except Exception:
            reason = "DEADLINE_UNAVAIL

### scripts/validate_development_head.py

"""Validate one captured HTTP/1.x header block; never make a network request.

This is a strict metadata subset, not a general HTTP client. Duplicate observed
fields and Transfer-Encoding are refused by policy, not by universal HTTP rules.
"""

import hashlib
import re

MAX_BYTES = 65536
MAX_CL_DIGITS = 19
MAX_CL_VALUE = 2**63 - 1


def _validate_label(value: str, name: str) -> None:
    if not isinstance(value, str) or not value:
        raise ValueError(f"{name} must be a non-empty string")
    if any(ord(c) < 32 or 127 <= ord(c) <= 159 for c in value):
        raise ValueError(f"{name} contains control characters")


def _parse_content_length(value: str) -> int:
    if not value:
        raise ValueError("Content-Length is blank")
    if len(value) > MAX_CL_DIGITS:
        raise ValueError("Content-Length has excessive digits")
    if not value.isascii() or not value.isdigit():
        raise ValueError("Content-Length contains non-ASCII digits or invalid characters")
    n = int(value)
    if n < 1:
        raise ValueError("Content-Length must be positive")
    if n > MAX_CL_VALUE:
        raise ValueError("Content-Length exceeds maximum")
    return n


def parse_head_evidence(raw_headers: bytes, *, url: str, observed_at: str) -> dict:
    """Return metadata or raise ValueError for an unsupported/invalid capture.

    URL/time are caller labels, not verified provenance. The size cap applies to
    this already-captured record; it proves no transport, deadline or RAM bound.
    body_bytes=0 describes this input only. Length/validators/header hash do not
    establish payload integrity, contents, decoded size or scientific acceptance.
    """
    if not isinstance(raw_headers, bytes):
        raise ValueError("raw_headers must be bytes")
    if len(raw_headers) > MAX_BYTES:
        raise ValueError("raw_headers exceeds maximum size")
    _validate_label(url, "url")
    _validate_label(observed_at, "observed_at")

    end = raw_headers.find(b"\r\n\r\n")
    if end < 0:
        raise ValueError("missing final CRLF CRLF terminator")
    if end != len(raw_headers) - 4:
        raise ValueError("trailing bytes after headers")

    lines = raw_headers[:end].split(b"\r\n")
    status = re.fullmatch(rb"HTTP/1\.[01] ([0-9]{3}) [\t\x20-\x7e\x80-\xff]*", lines[0])
    if status is None:
        raise ValueError("malformed status line")
    http_status = int(status[1])
    if http_status != 200:
        raise ValueError(f"non-200 status: {http_status}")

    headers: dict[str, list[str]] = {}
    for line in lines[1:]:
        if not line:
            raise ValueError("empty header line")
        if line.startswith(b" ") or line.startswith(b"\t"):
            raise ValueError("obsolete folded header")
        if b"\r" in line or b"\n" in line:
            raise ValueError("bare LF or CR in header")
        if b":" not in line:
            raise ValueError("malformed header line")
        name_bytes, value_bytes = line.split(b":", 1)
        if re.fullmatch(rb"[!#$%&'*+\-.^_`|~0-9A-Za-z]+", name_bytes) is None:
            raise ValueError(f"invalid header name: {name_bytes!r}")
        name = name_bytes.decode("ascii")
        value = value_bytes.decode("latin-1").strip(" \t")
        if any((ord(c) < 32 and c != "\t") or ord(c) == 127 for c in value):
            raise ValueError(f"control character in header value: {name!r}")
        headers.setdefault(name.lower(), []).append(value)

    if "transfer-encoding" in headers:
        raise ValueError("Transfer-Encoding is not allowed")

    cl_values = headers.get("content-length")
    if cl_values is None:
        raise ValueError("missing Content-Length")
    if len(cl_values) > 1:
        raise ValueError("duplicate Content-Length")
    if "," in cl_values[0]:
        raise ValueError("Content-Length contains comma list")
    declared_payload_bytes = _parse_content_length(cl_values[0])

    def _single(name: str) -> str | None:
        vals = headers.get(name)
        if vals is None

## 7. Next steps and stop conditions

| Step | Required evidence | What success permits |
| --- | --- | --- |
| E2-M1 | Reviewed native runtime/control design and bounded offline probe allocations | Implement the live binding |
| E2-M2 / M3 | Real enforcement of memory/no-swap, one total deadline, output limits, safe cleanup, and trusted TLS/input use | Independent controls review |
| E2-M4 | Accepted controls and the existing exact HEAD scope approval | One bounded metadata request only |
| E2-R / E3 | Applicable R workspace/assay reader proof, resource contract, and authorized inspection | Assess actual paired inputs |
| M1–M5 | Accepted release/QC, donor support, measured common regions, provenance, and frozen protocol | Consider the real internal training run |
| M6–M8 | Accepted development results, frozen model, and accepted external cohort | One-shot external evaluation |

Stop on missing evidence, changed hashes, failed controls, or unavailable resources. A successful HEAD does not prove valid RNA/ATAC assays. Do not weaken scientific gates to obtain a positive result.

### Existing implementation and detailed records

- [Full plan](../tasks/plan.md) and [ordered checklist](../tasks/todo.md)
- [Paired-data audit and current execution record](../docs/PAIRED_MULTIOME_AUDIT.md)
- [Dataset roles and experimental question](../docs/PAIRED_DS_MULTIOME_DATASET_OPTIONS.md)
- [RNA outcome and original executed-notebook links](../docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md)
- [RNA donor-influence results](../docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md)
- [Paired training guide](../docs/PAIRED_MULTIOME_TRAINING.md)
- [Bounded input audit](../scripts/audit_multiome.py), [paired loader](../src/p22/data/multiome.py), [synthetic benchmark](../scripts/train_multiome.py)
- [Fold training](../src/p22/eval/multiome_runner.py) and [frozen refit/scoring](../src/p22/eval/multiome_final.py)

These links expose existing work without silently running its acquisition or training commands.

In [9]:
notebook_decision = {
    "mode": "OFFLINE_EVIDENCE_AND_TESTS",
    "offline_tests_passed": OFFLINE_TESTS_PASSED,
    "source_status": contract["status"],
    "scientific_outcome": "INCONCLUSIVE",
    "live_requests_attempted": capture_refusal["requests_attempted"],
    "training_performed": False,
    "live_allowed": preflight_result["live_allowed"],
    "output_reserved": preflight_result["output_reserved"],
    "demo_header_is_synthetic": True,
    "next_action": "Review E2-M1 native controls and bounded offline probe allocations.",
}
assert not output_candidate.exists()
print(json.dumps(notebook_decision, indent=2))

{
  "mode": "OFFLINE_EVIDENCE_AND_TESTS",
  "offline_tests_passed": true,
  "source_status": "SOURCE_UNRESOLVED",
  "scientific_outcome": "INCONCLUSIVE",
  "live_requests_attempted": 0,
  "training_performed": false,
  "live_allowed": false,
  "output_reserved": false,
  "demo_header_is_synthetic": true,
  "next_action": "Review E2-M1 native controls and bounded offline probe allocations."
}
